# Hypothesis 5: sample efficiency and robustness

We compare the original features and Forest Sketch at three labeled-training fractions, then evaluate both on a noisy version of the same held-out test set. The evidence is exploratory: the test set is not used to select a fraction or method.

In [1]:
import warnings
from sklearn.random_projection import DataDimensionalityWarning
warnings.filterwarnings(
    "ignore",
    category=DataDimensionalityWarning,
    message="The number of components is higher than the number of features.*",
)

import sys
from pathlib import Path
notebooks_dir = Path.cwd() / 'notebooks'
if not (notebooks_dir / '_hypothesis_utils.py').exists(): notebooks_dir = Path.cwd()
sys.path.insert(0, str(notebooks_dir))
import numpy as np
import pandas as pd
from _hypothesis_utils import classification_data, classification_score, downstream_classifier, forest_sketch

In [2]:
rows = []
for seed in [0, 1]:
    X_train, _, X_test, y_train, _, y_test = classification_data(seed)
    dimension = 20 * X_train.shape[1]
    print(f'seed={seed}: p={X_train.shape[1]}, d=20*p={dimension}')
    for fraction in [0.25, 0.50, 1.00]:
        count = int(len(X_train) * fraction)
        X_subset, y_subset = X_train[:count], y_train[:count]
        baseline_accuracy, _ = classification_score(downstream_classifier(seed), X_subset, y_subset, X_test, y_test)
        sketch = forest_sketch(seed, dimension, 2)
        X_subset_view = sketch.fit_transform(X_subset, y_subset)
        X_test_view = sketch.transform(X_test)
        sketch_accuracy, _ = classification_score(downstream_classifier(seed), X_subset_view, y_subset, X_test_view, y_test)
        rows.extend([
            {'seed': seed, 'fraction': fraction, 'method': 'original', 'accuracy': baseline_accuracy},
            {'seed': seed, 'fraction': fraction, 'method': 'Forest Sketch', 'accuracy': sketch_accuracy},
        ])
results = pd.DataFrame(rows)
summary = results.groupby(['fraction', 'method']).accuracy.agg(['mean', 'std']).reset_index()
display(summary.round(3))

noise_rows = []
for seed in [0, 1]:
    X_train, _, X_test, y_train, _, y_test = classification_data(seed)
    dimension = 20 * X_train.shape[1]
    print(f'seed={seed}: p={X_train.shape[1]}, d=20*p={dimension}')
    rng = np.random.RandomState(seed)
    noisy_test = X_test + rng.normal(0, 0.10, size=X_test.shape)
    baseline_accuracy, _ = classification_score(downstream_classifier(seed), X_train, y_train, noisy_test, y_test)
    sketch = forest_sketch(seed, dimension, 2)
    train_view = sketch.fit_transform(X_train, y_train)
    noisy_view = sketch.transform(noisy_test)
    sketch_accuracy, _ = classification_score(downstream_classifier(seed), train_view, y_train, noisy_view, y_test)
    noise_rows.extend([{'seed': seed, 'method': 'original', 'accuracy': baseline_accuracy}, {'seed': seed, 'method': 'Forest Sketch', 'accuracy': sketch_accuracy}])
noise_summary = pd.DataFrame(noise_rows).groupby('method').accuracy.agg(['mean', 'std']).round(3)
display(noise_summary)
low_data = summary[summary.fraction == 0.25].set_index('method')
supported = low_data.loc['Forest Sketch', 'mean'] > low_data.loc['original', 'mean']
print(f'Hypothesis 5: {"SUPPORTED" if supported else "NOT SUPPORTED"} — low-data Forest Sketch advantage={low_data.loc["Forest Sketch", "mean"] - low_data.loc["original", "mean"]:.3f}')

seed=0: p=120, d=20*p=2400


seed=1: p=120, d=20*p=2400


,fraction,method,mean,std
0,0.25,Forest Sketch,0.770,0.003
1,0.25,original,0.627,0.035
2,0.50,Forest Sketch,0.786,0.010
3,0.50,original,0.652,0.058
4,1.00,Forest Sketch,0.830,0.029
5,1.00,original,0.650,0.042


seed=0: p=120, d=20*p=2400


seed=1: p=120, d=20*p=2400


,mean,std
method,,
Forest Sketch,0.825,0.029
original,0.653,0.047


Hypothesis 5: SUPPORTED — low-data Forest Sketch advantage=0.143


## Conclusion

**Hypothesis:** Forest Sketch is more sample-efficient and remains competitive under moderate feature noise.

**Experiment:** compare original features and two-iteration Forest Sketch at 25%, 50%, and 100% of the labeled training data, then evaluate both on a noisy held-out test set across two seeds.

**Measure:** held-out classification accuracy and integer error count, summarized by mean and standard deviation.

**Result:** not supported in this run: the low-data Forest Sketch accuracy was 0.008 below the original-feature baseline.